# OW-OVD Task 1 Experimentation Notebook
Notebook này được cấu hình để chạy lại Task 1 từ đầu với tập dữ liệu IP102 (đã lọc còn 25 lớp), trong đó Task 1 học **7 lớp** đầu tiên và huấn luyện trong **5 epoch** sử dụng cấu hình mới tại `configs/custom/ip102_t1.py`.

**Lưu ý**: Đã được tối ưu hóa chế độ offline-first để tự động tìm đường dẫn dataset, nạp trực tiếp weights YOLO-World và mô hình CLIP từ Kaggle Input nếu có, tự động tải về nếu chạy trực tuyến.

In [1]:
# Cell 1: Clone repository và thiết lập thư mục làm việc trên Kaggle
import os

repo_url = "https://github.com/nta2112/OW_OVD-An-custom.git"
working_dir = "/kaggle/working/OW_OVD"

if not os.path.exists(working_dir):
    print("Cloning repository...")
    !git clone {repo_url} {working_dir}
else:
    print("Repository đã tồn tại. Đang cập nhật (pull)...")
    %cd {working_dir}
    !git pull

%cd {working_dir}

# Clone thư mục con mmyolo vào third_party nếu chưa có
if not os.path.exists("third_party/mmyolo"):
    print("Cloning mmyolo...")
    !git clone https://github.com/open-mmlab/mmyolo.git third_party/mmyolo
else:
    print("mmyolo đã tồn tại.")

Cloning repository...
Cloning into '/kaggle/working/OW_OVD'...
remote: Enumerating objects: 834, done.
remote: Counting objects: 100% (207/207), done.
remote: Compressing objects: 100% (138/138), done.
remote: Total 834 (delta 127), reused 145 (delta 69), pack-reused 627 (from 1)
Receiving objects: 100% (834/834), 1.43 MiB | 9.74 MiB/s, done.
Resolving deltas: 100% (522/522), done.
/kaggle/working/OW_OVD
Cloning mmyolo...
Cloning into 'third_party/mmyolo'...
remote: Enumerating objects: 4968, done.
remote: Counting objects: 100% (1341/1341), done.
remote: Compressing objects: 100% (294/294), done.
remote: Total 4968 (delta 1133), reused 1047 (delta 1047), pack-reused 3627 (from 1)
Receiving objects: 100% (4968/4968), 3.62 MiB | 16.62 MiB/s, done.
Resolving deltas: 100% (3216/3216), done.


In [2]:
# Cell 2: Cài đặt toàn bộ môi trường và các thư viện cần thiết
# 1. Hạ cấp PyTorch & Torchvision xuống bản ổn định 2.4.0
!pip install torch==2.4.0+cu121 torchvision==0.19.0+cu121 --extra-index-url https://download.pytorch.org/whl/cu121

# 2. Cài đặt MMCV từ bản build sẵn tương thích
!pip install mmcv -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html

# 3. Cài đặt các thư viện bổ trợ khác
!pip install matplotlib pycocotools terminaltables mmengine prettytable wcwidth open_clip_torch transformers

# 4. Cài đặt MMDetection và MMYOLO
!pip install "mmdet>=3.1.0" --no-deps
!pip install --no-build-isolation --no-deps third_party/mmyolo

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 86.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 70.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 34.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 81.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 9.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 33.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 14.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# Cell 3: Vá lỗi giới hạn phiên bản MMCV trong mmdet và mmyolo
import importlib.util
import sys

packages = ['mmdet', 'mmyolo']
for pkg in packages:
    spec = importlib.util.find_spec(pkg)
    if spec and spec.origin:
        try:
            with open(spec.origin, 'r') as f:
                content = f.read()
            # Thay thế kiểm tra phiên bản để tránh crash
            new_content = content.replace('2.1.0', '2.3.0').replace('2.2.0', '2.3.0')
            with open(spec.origin, 'w') as f:
                f.write(new_content)
            print(f"-> Đã vá lỗi phiên bản MMCV cho {pkg} thành công.")
        except Exception as e:
            print(f"-> Lỗi khi vá {pkg}: {e}")

-> Đã vá lỗi phiên bản MMCV cho mmdet thành công.
-> Đã vá lỗi phiên bản MMCV cho mmyolo thành công.


In [4]:
# # Cell 4: Tạo cấu hình ip102_t1.py ngay trên Kaggle
# import os

# os.makedirs('configs/custom', exist_ok=True)
# config_content = """_base_ = ('../../third_party/mmyolo/configs/yolov8/'
#           'yolov8_l_syncbn_fast_8xb16-500e_coco.py')
# custom_imports = dict(imports=['yolo_world'], allow_failed_imports=False)

# # Suppress MMEngine's extremely verbose configuration printing at startup
# import mmengine
# try:
#     mmengine.Config.pretty_text = property(lambda self: "[Config dump suppressed for cleaner logs]")
#     from mmengine.logging import MMLogger
#     _orig_info = MMLogger.info
#     def _clean_info(self, msg, *args, **kwargs):
#         msg_str = str(msg)
#         if any(term in msg_str for term in ['OurDetector(', 'MultiModalYOLOBackbone(', 'paramwise_options', 'Checkpoints will be saved to']):
#             return
#         _orig_info(self, msg, *args, **kwargs)
#     MMLogger.info = _clean_info
# except Exception:
#     pass

# # Fool-proof monkey patch to fix double 'test/test/' path bug in Kaggle datasets
# try:
#     from mmyolo.datasets import YOLOv5CocoDataset
#     _orig_parse = YOLOv5CocoDataset.parse_data_info
#     def _patched_parse(self, raw_data_info):
#         data_info = _orig_parse(self, raw_data_info)
#         if 'img_path' in data_info and 'test/test/' in data_info['img_path']:
#             data_info['img_path'] = data_info['img_path'].replace('test/test/', 'test/')
#         return data_info
#     YOLOv5CocoDataset.parse_data_info = _patched_parse
# except Exception:
#     pass

# # Tự động tìm đường dẫn dataset trên Kaggle
# import glob
# import os
# import json

# dataset_root = None
# for path in [
#     '/kaggle/input/datasets/nta212/ip102-for-object-detection',
#     '/kaggle/input/ip102-for-object-detection',
#     'data/IP102',
#     '.'
# ]:
#     if os.path.exists(os.path.join(path, 'train.json')):
#         dataset_root = path
#         break

# if dataset_root is None:
#     paths = glob.glob('/kaggle/input/**/train.json', recursive=True)
#     if paths:
#         dataset_root = os.path.dirname(paths[0])

# if dataset_root is None:
#     dataset_root = '.'  # Fallback local path

# train_json = os.path.join(dataset_root, 'train.json')
# test_json = os.path.join(dataset_root, 'test.json')
# val_json = os.path.join(dataset_root, 'val.json')

# # Tự động quét tìm thư mục thực sự chứa ảnh .jpg để tránh FileNotFoundError
# image_data_root = None
# for root, dirs, files in os.walk(dataset_root):
#     if any(f.lower().endswith('.jpg') for f in files):
#         image_data_root = root
#         break
# if image_data_root is None:
#     image_data_root = dataset_root

# # Dynamically load class names from IP102 annotations
# class_names = None
# try:
#     with open(train_json, 'r') as f:
#         coco_data = json.load(f)
#     categories = sorted(coco_data['categories'], key=lambda x: x['id'])
#     class_names = [cat['name'] for cat in categories]
# except Exception:
#     pass

# if class_names is None:
#     class_names = ['14', '15', '16', '18', '22', '23', '24', '25', '26', '37', '38', '39', '45', '46', '47', '48', '49', '50', '51', '66', '67', '69', '70', '86', '101']

# # open world setting
# prev_intro_cls = 0
# cur_intro_cls = 7
# embedding_path = 'data/IP102/ip102_gt_embeddings.npy'
# att_embeddings = 'data/IP102/task_att_1_embeddings.pth'
# pipline = [dict(type='att_select', log_start_epoch=1)]
# thr = 0.55
# alpha = 0.2
# use_sigmoid = True
# distributions = 'data/IP102/mowod_distribution_sim1.pth'
# top_k = 10

# # yolo world setting
# num_classes = 7
# num_training_classes = 7
# max_epochs = 5
# close_mosaic_epochs = max_epochs
# save_epoch_intervals = 1
# text_channels = 512
# neck_embed_channels = [128, 256, _base_.last_stage_out_channels // 2]
# neck_num_heads = [4, 8, _base_.last_stage_out_channels // 2 // 32]
# base_lr = 1e-4
# weight_decay = 0.05
# train_batch_size_per_gpu = 24
# load_from = 'pretrained_models/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth'
# persistent_workers = True

# # model settings
# model = dict(type='OurDetector',
#              mm_neck=True,
#              num_train_classes=num_training_classes,
#              num_test_classes=num_classes,
#              embedding_path=embedding_path,
#              prompt_dim=text_channels,
#              num_prompts=len(class_names),
#              pipline=pipline,
#              data_preprocessor=dict(type='YOLOv5DetDataPreprocessor'),
#              backbone=dict(_delete_=True,
#                            type='MultiModalYOLOBackbone',
#                            text_model=None,
#                            image_model={{_base_.model.backbone}},
#                            frozen_stages=4,
#                            with_text_model=False),
#              neck=dict(type='YOLOWorldPAFPN',
#                        freeze_all=False,
#                        guide_channels=text_channels,
#                        embed_channels=neck_embed_channels,
#                        num_heads=neck_num_heads,
#                        block_cfg=dict(type='MaxSigmoidCSPLayerWithTwoConv')),
#              bbox_head=dict(type='OurHead',
#                             att_embeddings=att_embeddings,
#                             thr=thr,
#                             alpha=alpha,
#                             use_sigmoid=use_sigmoid,
#                             distributions=distributions,
#                             prev_intro_cls=prev_intro_cls,
#                             cur_intro_cls=cur_intro_cls,
#                             top_k=top_k,
#                             head_module=dict(
#                                 type='OurHeadModule',
#                                 freeze_all=False,
#                                 use_bn_head=True,
#                                 embed_dims=text_channels,
#                                 num_classes=num_training_classes,),
#                             ),
#              train_cfg=dict(assigner=dict(num_classes=num_training_classes)))

# # dataset settings
# coco_train_dataset = dict(
#         _delete_=True,
#         type='MultiModalDataset',
#         dataset=dict(
#             type='YOLOv5CocoDataset',
#             metainfo=dict(classes=class_names[:7]),
#             data_root=image_data_root,
#             ann_file=train_json,
#             data_prefix=dict(img=''),
#             filter_cfg=dict(filter_empty_gt=True, min_size=32)),
#         class_text_path='data/texts/IP102/class_texts.json',
#         pipeline=_base_.train_pipeline)

# train_dataloader = dict(persistent_workers=persistent_workers,
#                         batch_size=train_batch_size_per_gpu,
#                         num_workers=4,
#                         pin_memory=True,
#                         collate_fn=dict(type='yolow_collate'),
#                         dataset=coco_train_dataset)

# custom_hooks = [
#     dict(type='mmdet.PipelineSwitchHook',
#          switch_epoch=max_epochs - close_mosaic_epochs,
#          switch_pipeline=_base_.train_pipeline_stage2),
#     dict(type='OurWorkPiplineHook'),
#     dict(type='EarlyStoppingHook',
#          monitor='coco/Current class AP50',
#          rule='greater',
#          patience=2,
#          min_delta=0.001)
# ]

# default_hooks = dict(
#     checkpoint=dict(
#         interval=save_epoch_intervals, max_keep_ckpts=2, save_best='coco/Current class AP50', rule='greater',
#         type='CheckpointHook'),
#     logger=dict(interval=50, type='LoggerHook'),
#     param_scheduler=dict(
#         lr_factor=0.01,
#         max_epochs=max_epochs,
#         scheduler_type='linear',
#         type='YOLOv5ParamSchedulerHook'),
#     sampler_seed=dict(type='DistSamplerSeedHook'),
#     timer=dict(type='IterTimerHook'),
#     visualization=dict(type='mmdet.DetVisualizationHook'))

# train_cfg = dict(max_epochs=max_epochs,
#                  val_interval=999,
#                  dynamic_intervals=[(2, 1)])

# optim_wrapper = dict(
#     type='AmpOptimWrapper',
#     optimizer=dict(
#         _delete_=True,
#         type='AdamW',
#         lr=base_lr,
#         weight_decay=weight_decay,
#         batch_size_per_gpu=train_batch_size_per_gpu),
#     paramwise_cfg=dict(bias_decay_mult=0.0,
#                        norm_decay_mult=0.0,
#                        custom_keys={
#                            'backbone.text_model':
#                            dict(lr_mult=0.01),
#                            'logit_scale':
#                            dict(weight_decay=0.0),
#                            'embeddings':
#                            dict(weight_decay=0.0)
#                        }),
#     constructor='YOLOWv5OptimizerConstructor')

# test_pipeline = [
#     *_base_.test_pipeline[:-1],
#     dict(type='mmdet.PackDetInputs',
#          meta_keys=('img_id', 'img_path', 'ori_shape', 'img_shape',
#                     'scale_factor', 'pad_param'))
# ]

# test_dataloader = dict(
#     _delete_=True,
#     batch_size=24,
#     num_workers=4,
#     persistent_workers=True,
#     pin_memory=True,
#     drop_last=False,
#     sampler=dict(type='DefaultSampler', shuffle=False),
#     dataset=dict(type='YOLOv5CocoDataset',
#                  metainfo=dict(classes=class_names),
#                  data_root=image_data_root,
#                  ann_file=test_json,
#                  data_prefix=dict(img=''),
#                  filter_cfg=dict(filter_empty_gt=True, min_size=32),
#                  pipeline=test_pipeline)
# )

# test_evaluator = dict(_delete_=True,
#                       type='OWODEvaluator',
#                       prefix='coco',
#                       cfg=dict(
#                          dataset_root='data/IP102/voc/',
#                          ann_file=test_json,
#                          file_name='mowod/all_task_test.txt',
#                          prev_intro_cls=prev_intro_cls,
#                          cur_intro_cls=cur_intro_cls,
#                          unknown_id=25,
#                          class_names=class_names
#                       )
#                      )

# val_dataloader = dict(
#     _delete_=True,
#     batch_size=24,
#     num_workers=4,
#     persistent_workers=True,
#     pin_memory=True,
#     drop_last=False,
#     sampler=dict(type='DefaultSampler', shuffle=False),
#     dataset=dict(type='YOLOv5CocoDataset',
#                  metainfo=dict(classes=class_names),
#                  data_root=image_data_root,
#                  ann_file=val_json,
#                  data_prefix=dict(img=''),
#                  filter_cfg=dict(filter_empty_gt=True, min_size=32),
#                  pipeline=test_pipeline)
# )

# val_evaluator = dict(_delete_=True,
#                       type='OWODEvaluator',
#                       prefix='coco',
#                       cfg=dict(
#                          dataset_root='data/IP102/voc_val/',
#                          ann_file=val_json,
#                          file_name='mowod/all_task_val.txt',
#                          prev_intro_cls=prev_intro_cls,
#                          cur_intro_cls=cur_intro_cls,
#                          unknown_id=25,
#                          class_names=class_names
#                       )
#                      )
# find_unused_parameters = True

# # Clean up all temporary variables from config namespace to avoid deepcopy/pickle errors (e.g. TextIOWrapper)
# for var in ['json', 'os', 'glob', 'path', 'paths', 'f', 'coco_data', 'categories', 'dataset_root', 'train_json', 'test_json', 'val_json', 'image_data_root']:
#     globals().pop(var, None)
# """

# with open('configs/custom/ip102_t1.py', 'w', encoding='utf-8') as f:
#     f.write(config_content)
# print("-> Đã ghi file cấu hình configs/custom/ip102_t1.py thành công!")

In [5]:
# Cell 5: Tạo thư mục, tải/nạp weights và sinh embeddings + XML annotations
import json
import torch
import numpy as np
import os
import glob

# Triệt để monkeypatch check_torch_load_is_safe từ đầu để tránh lỗi PyTorch 2.6 / CVE-2025-32434
import transformers.utils.import_utils
import transformers.utils
import transformers.modeling_utils
transformers.utils.import_utils.check_torch_load_is_safe = lambda: None
transformers.utils.check_torch_load_is_safe = lambda: None
transformers.modeling_utils.check_torch_load_is_safe = lambda: None

from transformers import AutoTokenizer, CLIPTextModelWithProjection

# 1. Tạo thư mục
os.makedirs('pretrained_models', exist_ok=True)
os.makedirs('pretrained_models/clip-vit-base-patch32', exist_ok=True)
os.makedirs('data/IP102', exist_ok=True)
os.makedirs('data/texts/IP102', exist_ok=True)

# 2. Tải/nạp pretrain weights của YOLO-World từ Kaggle Input
weights_path = 'pretrained_models/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth'
if not os.path.exists(weights_path):
    print("-> Đang tìm weights YOLO-World trong Kaggle Input...")
    kaggle_input_weights = glob.glob('/kaggle/input/models/nta212/yolo-world/pytorch/default/1/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth', recursive=True)
    if kaggle_input_weights:
        !cp {kaggle_input_weights[0]} pretrained_models/
        print("-> Đã nạp weights YOLO-World thành công!")
    else:
        print("-> Không tìm thấy weights cục bộ. Đang tải về từ HF...")
        !wget -O {weights_path} https://huggingface.co/wondervictor/YOLO-World/resolve/main/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth
else:
    print("-> Pretrained weights đã tồn tại.")

# 3. Đọc danh sách 25 lớp mới (Tự động tìm kiếm đường dẫn dataset trên Kaggle)
print("-> Đang định vị dataset trên Kaggle...")
dataset_root = None
for path in [
    '/kaggle/input/datasets/nta212/ip102-for-object-detection',
    '/kaggle/input/ip102-for-object-detection',
    'data/IP102',
    '.'
]:
    if os.path.exists(os.path.join(path, 'train.json')):
        dataset_root = path
        break

if dataset_root is None:
    paths = glob.glob('/kaggle/input/**/train.json', recursive=True)
    if paths:
        dataset_root = os.path.dirname(paths[0])

if dataset_root is None:
    raise FileNotFoundError("Không thể tìm thấy train.json của dataset IP102 trong Kaggle Input!")

train_json = os.path.join(dataset_root, 'train.json')
print(f"-> Đã phát hiện dataset train.json tại: {train_json}")

with open(train_json, 'r') as f:
    coco_data = json.load(f)

categories = sorted(coco_data['categories'], key=lambda x: x['id'])
class_names = [cat['name'] for cat in categories]
num_classes = len(class_names)
print(f"-> Tổng số lớp học: {num_classes}")
print(f"-> Danh sách lớp: {class_names}")

# 4. Lưu class_texts.json
class_texts = [[name] for name in class_names]
with open('data/texts/IP102/class_texts.json', 'w') as f:
    json.dump(class_texts, f)

# 5. Nạp CLIP Model cục bộ từ Kaggle Input (Chế độ Offline-first)
print("-> Đang tìm kiếm CLIP model trong Kaggle Input...")
clip_local_dir = 'pretrained_models/clip-vit-base-patch32'

clip_configs = glob.glob('/kaggle/input/models/yujkaggle/openaiclip-vit-base-patch32/pytorch/default/1/config.json', recursive=True)
kaggle_input_clip = None
for cfg in clip_configs:
    if 'clip-vit-base-patch32' in cfg or 'vit-base-patch32' in cfg:
        kaggle_input_clip = os.path.dirname(cfg)
        break

if kaggle_input_clip:
    print(f"-> Tìm thấy CLIP model tại: {kaggle_input_clip}")
    !cp -rf {kaggle_input_clip}/* pretrained_models/clip-vit-base-patch32/
    print("-> Đã nạp CLIP model cục bộ thành công!")
    tokenizer = AutoTokenizer.from_pretrained(clip_local_dir)
    model = CLIPTextModelWithProjection.from_pretrained(clip_local_dir, use_safetensors=False, weights_only=False)
else:
    print("-> Không tìm thấy CLIP model cục bộ. Đang tải tự động trực tuyến...")
    clip_local_dir = 'openai/clip-vit-base-patch32'
    tokenizer = AutoTokenizer.from_pretrained(clip_local_dir)
    model = CLIPTextModelWithProjection.from_pretrained(clip_local_dir, use_safetensors=True)

print("-> Đang sinh class embeddings từ CLIP...")
model.eval()
embeddings = []
with torch.no_grad():
    for name in class_names:
        inputs = tokenizer(name, padding=True, return_tensors="pt")
        outputs = model(**inputs)
        embed = outputs.text_embeds[0].cpu().numpy()
        embed = embed / np.linalg.norm(embed)
        embeddings.append(embed)

np.save('data/IP102/ip102_gt_embeddings.npy', np.array(embeddings))
print("-> Đã lưu ip102_gt_embeddings.npy.")

# 6. Sinh dummy attribute embeddings
num_att = num_classes * 25
torch.save({
    'att_embedding': torch.zeros(num_att, 512),
    'att_text': [f"att_{i}" for i in range(num_att)]
}, 'data/IP102/task_att_1_embeddings.pth')

# 7. Sinh dummy distributions
thrs = [0.55]
pos_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
neg_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
torch.save({
    'positive_distributions': pos_dist,
    'negative_distributions': neg_dist
}, 'data/IP102/mowod_distribution_sim1.pth')
print("-> Đã sinh attribute embeddings và distributions.")

-> Đang tìm weights YOLO-World trong Kaggle Input...
-> Đã nạp weights YOLO-World thành công!
-> Đang định vị dataset trên Kaggle...
-> Đã phát hiện dataset train.json tại: /kaggle/input/datasets/nta212/ip102-for-object-detection/train.json
-> Tổng số lớp học: 25
-> Danh sách lớp: ['14', '15', '16', '18', '22', '23', '24', '25', '26', '37', '38', '39', '45', '46', '47', '48', '49', '50', '51', '66', '67', '69', '70', '86', '101']
-> Đang tìm kiếm CLIP model trong Kaggle Input...
-> Tìm thấy CLIP model tại: /kaggle/input/models/yujkaggle/openaiclip-vit-base-patch32/pytorch/default/1
-> Đã nạp CLIP model cục bộ thành công!


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: pretrained_models/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.embeddings.class_embedding                        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model

-> Đang sinh class embeddings từ CLIP...
-> Đã lưu ip102_gt_embeddings.npy.
-> Đã sinh attribute embeddings và distributions.


# task 1

In [6]:
# # Cell 6: Chạy huấn luyện Task 1 (Sử dụng torchrun đa GPU)
# # Lưu ý: configs/custom/ip102_t1.py đã cấu hình học 7 lớp, max_epochs = 5
# !PYTHONPATH=. torchrun --nproc_per_node=2 third_party/mmyolo/tools/train.py configs/custom/ip102_t1.py --launcher pytorch

In [7]:
# # Cell 7: Kiểm tra và đánh giá mô hình tốt nhất sau khi huấn luyện xong
# import glob
# import os

# # Tìm checkpoint tốt nhất
# weights = glob.glob('work_dirs/ip102_t1/best_*.pth')
# if weights:
#     best_model_path = weights[0]
#     print(f"-> Đã tìm thấy mô hình tốt nhất tại: {best_model_path}")
#     # Đánh giá kiểm thử
#     !PYTHONPATH=. python third_party/mmyolo/tools/test.py configs/custom/ip102_t1.py "{best_model_path}"
# else:
#     print("-> Không tìm thấy file weight tốt nhất. Có thể quá trình huấn luyện chưa kết thúc hoặc xảy ra lỗi!")

# task 2

In [8]:
# 1. Tạo liên kết mềm (symlink) không chứa khoảng trắng trỏ tới checkpoint gốc
!ln -sf "/kaggle/input/models/nta212/task-1-ow-ovd-25-class/pytorch/default/1/best_coco_Current class AP50_epoch_5.pth" /kaggle/working/best_t1.pth

# 2. Chạy huấn luyện Task 2 sử dụng đường dẫn liên kết mềm vừa tạo
!PYTHONPATH=. torchrun --nproc_per_node=2 third_party/mmyolo/tools/train.py \
    configs/custom/ip102_t2.py \
    --launcher pytorch \
    --cfg-options load_from="/kaggle/working/best_t1.pth"


W0719 16:27:55.951000 135668560434304 torch/distributed/run.py:779] 
W0719 16:27:55.951000 135668560434304 torch/distributed/run.py:779] *****************************************
W0719 16:27:55.951000 135668560434304 torch/distributed/run.py:779] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0719 16:27:55.951000 135668560434304 torch/distributed/run.py:779] *****************************************
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` 

In [9]:
# Cell đánh giá Task 2 sau khi huấn luyện xong
import glob
best_t2 = glob.glob('work_dirs/ip102_t2/best_*.pth')[0]
!PYTHONPATH=. python third_party/mmyolo/tools/test.py configs/custom/ip102_t2.py "{best_t2}"
# Chạy đánh giá Task 2 (Sử dụng symlink vừa tạo)
# Chạy đánh giá Task 2 (Sử dụng symlink vừa tạo)


/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
07/19 16:44:22 - mmengine - WARNING - Failed to search registry with scope "mmyolo" in the "log_processor" registry tree. As a workaround, the current "log_processor" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmyolo" is a correct scope, or whether the registry is initialized.
07/19 16:44:23 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 1959189394
    GPU 0,1: Tesla T4
    

In [10]:
!zip -r /kaggle/working/ow_ovd_task_2.zip work_dirs 

  adding: work_dirs/ (stored 0%)
  adding: work_dirs/ip102_t2/ (stored 0%)
  adding: work_dirs/ip102_t2/20260719_164422/ (stored 0%)
  adding: work_dirs/ip102_t2/20260719_164422/20260719_164422.log (deflated 69%)
  adding: work_dirs/ip102_t2/20260719_164422/vis_data/ (stored 0%)
  adding: work_dirs/ip102_t2/20260719_164422/vis_data/config.py (stored 0%)
  adding: work_dirs/ip102_t2/20260719_164422/20260719_164422.json (deflated 62%)
  adding: work_dirs/ip102_t2/best_coco_Known AP50_epoch_5.pth (deflated 8%)
  adding: work_dirs/ip102_t2/20260719_162919/ (stored 0%)
  adding: work_dirs/ip102_t2/20260719_162919/vis_data/ (stored 0%)
  adding: work_dirs/ip102_t2/20260719_162919/vis_data/config.py (stored 0%)
  adding: work_dirs/ip102_t2/20260719_162919/vis_data/20260719_162919.json (deflated 69%)
  adding: work_dirs/ip102_t2/20260719_162919/vis_data/scalars.json (deflated 69%)
  adding: work_dirs/ip102_t2/20260719_162919/20260719_162919.log (deflated 92%)
  adding: work_dirs/ip102_t2/ip102

# task 3

In [11]:
# Cell chạy huấn luyện Task 3 (Sử dụng copy thay vì symlink)
import glob
import shutil

# 1. Tìm và copy checkpoint Task 2
best_t2_paths = glob.glob('work_dirs/ip102_t2/best_*.pth')
if not best_t2_paths:
    raise FileNotFoundError("Không tìm thấy checkpoint Task 2 trong work_dirs/ip102_t2/!")

shutil.copy2(best_t2_paths[0], '/kaggle/working/best_t2.pth')
print(f"-> Đã copy checkpoint Task 2 thành công từ {best_t2_paths[0]}")

# 2. Chạy train Task 3 dùng file đã copy (không có khoảng trắng trong đường dẫn)
!PYTHONPATH=. torchrun --nproc_per_node=2 third_party/mmyolo/tools/train.py \
    configs/custom/ip102_t3.py \
    --launcher pytorch \
    --cfg-options load_from="/kaggle/working/best_t2.pth"


-> Đã copy checkpoint Task 2 thành công từ work_dirs/ip102_t2/best_coco_Known AP50_epoch_5.pth
W0719 16:48:37.088000 133749449655424 torch/distributed/run.py:779] 
W0719 16:48:37.088000 133749449655424 torch/distributed/run.py:779] *****************************************
W0719 16:48:37.088000 133749449655424 torch/distributed/run.py:779] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0719 16:48:37.088000 133749449655424 torch/distributed/run.py:779] *****************************************
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
/usr/local/lib/python3.12/

In [12]:
# Cell đánh giá Task 3 sau khi huấn luyện xong
import glob
import shutil

# 1. Tìm và copy checkpoint Task 3 vừa huấn luyện xong
best_t3_paths = glob.glob('work_dirs/ip102_t3/best_*.pth')
if not best_t3_paths:
    raise FileNotFoundError("Không tìm thấy checkpoint Task 3 trong work_dirs/ip102_t3/!")

shutil.copy2(best_t3_paths[0], '/kaggle/working/best_t3.pth')
print(f"-> Đã copy checkpoint Task 3 thành công từ {best_t3_paths[0]}")

# 2. Tiến hành test mô hình dùng file checkpoint đã copy
!PYTHONPATH=. python third_party/mmyolo/tools/test.py \
    configs/custom/ip102_t3.py \
    /kaggle/working/best_t3.pth


-> Đã copy checkpoint Task 3 thành công từ work_dirs/ip102_t3/best_coco_Known AP50_epoch_5.pth
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
07/19 17:07:50 - mmengine - WARNING - Failed to search registry with scope "mmyolo" in the "log_processor" registry tree. As a workaround, the current "log_processor" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmyolo" is a correct scope, or whether the registry is initialized.
07/19 17:07:51 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
    CUDA availab

# task 4

In [13]:
# Cell chạy huấn luyện Task 4 (Sử dụng copy thay vì symlink)
import glob
import shutil

# 1. Tìm và copy checkpoint Task 3 làm tiền đề (load_from) cho Task 4
best_t3_paths = glob.glob('work_dirs/ip102_t3/best_*.pth')
if not best_t3_paths:
    raise FileNotFoundError("Không tìm thấy checkpoint Task 3 trong work_dirs/ip102_t3/!")

shutil.copy2(best_t3_paths[0], '/kaggle/working/best_t3.pth')
print(f"-> Đã copy checkpoint Task 3 thành công từ {best_t3_paths[0]}")

# 2. Chạy train Task 4 dùng file đã copy
!PYTHONPATH=. torchrun --nproc_per_node=2 third_party/mmyolo/tools/train.py \
    configs/custom/ip102_t4.py \
    --launcher pytorch \
    --cfg-options load_from="/kaggle/working/best_t3.pth"


-> Đã copy checkpoint Task 3 thành công từ work_dirs/ip102_t3/best_coco_Known AP50_epoch_5.pth
W0719 17:11:03.364000 136406651135104 torch/distributed/run.py:779] 
W0719 17:11:03.364000 136406651135104 torch/distributed/run.py:779] *****************************************
W0719 17:11:03.364000 136406651135104 torch/distributed/run.py:779] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0719 17:11:03.364000 136406651135104 torch/distributed/run.py:779] *****************************************
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
/usr/local/lib/python3.12/

In [14]:
# Cell đánh giá Task 4 sau khi huấn luyện xong
import glob
import shutil

# 1. Tìm và copy checkpoint Task 4 vừa huấn luyện xong
best_t4_paths = glob.glob('work_dirs/ip102_t4/best_*.pth')
if not best_t4_paths:
    raise FileNotFoundError("Không tìm thấy checkpoint Task 4 trong work_dirs/ip102_t4/!")

shutil.copy2(best_t4_paths[0], '/kaggle/working/best_t4.pth')
print(f"-> Đã copy checkpoint Task 4 thành công từ {best_t4_paths[0]}")

# 2. Tiến hành test mô hình dùng file checkpoint đã copy
!PYTHONPATH=. python third_party/mmyolo/tools/test.py \
    configs/custom/ip102_t4.py \
    /kaggle/working/best_t4.pth


-> Đã copy checkpoint Task 4 thành công từ work_dirs/ip102_t4/best_coco_Known AP50_epoch_5.pth
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
07/19 17:37:03 - mmengine - WARNING - Failed to search registry with scope "mmyolo" in the "log_processor" registry tree. As a workaround, the current "log_processor" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmyolo" is a correct scope, or whether the registry is initialized.
07/19 17:37:03 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
    CUDA availab

In [15]:
# Nén toàn bộ kết quả của 4 task lại thành file zip
!zip -r /kaggle/working/ow_ovd_results.zip work_dirs

  adding: work_dirs/ (stored 0%)
  adding: work_dirs/ip102_t4/ (stored 0%)
  adding: work_dirs/ip102_t4/20260719_171202/ (stored 0%)
  adding: work_dirs/ip102_t4/20260719_171202/vis_data/ (stored 0%)
  adding: work_dirs/ip102_t4/20260719_171202/vis_data/20260719_171202.json (deflated 70%)
  adding: work_dirs/ip102_t4/20260719_171202/vis_data/config.py (stored 0%)
  adding: work_dirs/ip102_t4/20260719_171202/vis_data/scalars.json (deflated 70%)
  adding: work_dirs/ip102_t4/20260719_171202/20260719_171202.log (deflated 92%)
  adding: work_dirs/ip102_t4/best_coco_Known AP50_epoch_5.pth (deflated 8%)
  adding: work_dirs/ip102_t4/20260719_173703/ (stored 0%)
  adding: work_dirs/ip102_t4/20260719_173703/20260719_173703.log (deflated 70%)
  adding: work_dirs/ip102_t4/20260719_173703/vis_data/ (stored 0%)
  adding: work_dirs/ip102_t4/20260719_173703/vis_data/config.py (stored 0%)
  adding: work_dirs/ip102_t4/20260719_173703/20260719_173703.json (deflated 66%)
  adding: work_dirs/ip102_t4/ip102

# lệnh để tìm file từ kết quả notebook cũ

In [16]:
# # /kaggle/input/notebooks/nta212/notebookb2ef849302/OW_OVD/work_dirs/ip102_t1/best_coco_Current class AP50_epoch_5
# import os
# import shutil
# from IPython.display import FileLink

# # --- CẤU HÌNH ---
# source_path = '/kaggle/input/notebooks/nta212/notebookb2ef849302/OW_OVD/work_dirs/ip102_t1/best_coco_Current class AP50_epoch_5.pth'
# working_dir = '/kaggle/working'
# filename = os.path.basename(source_path)
# destination_path = os.path.join(working_dir, filename)

# # 1. Kiểm tra file có tồn tại không
# if os.path.exists(source_path):
#     print(f"File tồn tại! Đang sao chép vào {working_dir}...")
    
#     # 2. Copy file vào thư mục làm việc
#     shutil.copy(source_path, destination_path)
#     print("Đã copy thành công.")
    
#     # 3. Tạo link tải về tự động
#     display(FileLink(destination_path))
# else:
#     print("File không tồn tại tại đường dẫn đã chỉ định.")

In [17]:
# !find /kaggle/input/notebooks/nta212/notebookb2ef849302 -name "best_coco_Current class AP50_epoch_5.pth"